In [2]:
import numpy as np
import time
import torch
import torch.autograd.profiler as profiler
import matplotlib.pyplot as plt

import sys
from pathlib import Path

# Set project root to one directory above the notebook
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src_dev.mpdo_circuit import MPDOCircuit
from src_dev.mpdo_torch import StateCreator, MPDOtorch
from src_dev.utils import BosonOperatorsTorch, eye_like
from src_dev.create_circuit import create_haar_random_circuit, dephasing_krauss_ops

from src_dev.svd_trunc import svd_trunc
from src_dev.utils import irescale, iregroup, sqrtm


In [3]:


device = "cuda:1"
num_channels = 2 # number of channels
num_layers = 1 # layer depth
options = {'max_BD': 500, 'max_PD': 100, 'cutoff_BD': 1e-6, 'cutoff_PD': 1e-4} # options
verbose = True

gamma = 0. # error rate

####################################
# INITIAL STATE
####################################

dict_rhos = {}

# initialize input state
state_creator = StateCreator(Nmax=1, num_batch=1)
list_ten = state_creator.product_state_fock([0] * num_channels, device=device)
rho = MPDOtorch(list_ten)

####################################
# Create circuit and run
####################################

circuit = create_haar_random_circuit(
    num_layers = num_layers,
    num_channels = num_channels,
    gamma = gamma,
    device=device
)

circuit.run(rho, options=options, verbose=verbose)

print(f"BDs: {rho.get_BDs()}")
print(f"PDs: {rho.get_PDs()}")

S_vN = rho.entropy_profile(entropy='entanglement')
S_P = rho.entropy_profile(entropy='purity')

print(f"S_vN: {S_vN}")
print(f"SP: {S_P}")



 iter (0,0)
tensor(0.9113, device='cuda:1', dtype=torch.float64)
[torch.Size([1, 1, 2, 2]), torch.Size([1, 2, 2, 1])]
BDs: [1, 2, 1]
PDs: [1, 1]
S_vN: tensor([-1.1102e-15,  3.9904e-01, -0.0000e+00], device='cuda:1',
       dtype=torch.float64)
SP: tensor([-1.1102e-15, -1.1102e-15], device='cuda:1', dtype=torch.float64)


In [4]:
gamma = 0.1

Ks = dephasing_krauss_ops(gamma=gamma, device=rho.device)

rho.krauss_dissipation(Ks[1:3])

S_vN = rho.entropy_profile(entropy='entanglement')
S_P = rho.entropy_profile(entropy='purity')

print(f"S_vN: {S_vN}")
print(f"S_purity: {S_P}")



S_vN: tensor([-1.1102e-15,  3.9904e-01, -0.0000e+00], device='cuda:1',
       dtype=torch.float64)
S_purity: tensor([0.5580, 0.4352], device='cuda:1', dtype=torch.float64)


In [5]:
import torch

probs = torch.tensor([0.1, 0.3, 0.05, 0.55])  # N = 4
n = 2

sample_list = []
for i in range(100):
    samples = torch.multinomial(probs, n, replacement=False)
    sample_list.append(samples)


print([s for s in sample_list])
print([s for s in sample_list if 2 in s])

[tensor([3, 2]), tensor([3, 0]), tensor([0, 1]), tensor([1, 3]), tensor([3, 2]), tensor([1, 3]), tensor([3, 1]), tensor([3, 1]), tensor([3, 1]), tensor([1, 0]), tensor([1, 3]), tensor([1, 0]), tensor([2, 3]), tensor([1, 3]), tensor([3, 1]), tensor([0, 1]), tensor([3, 1]), tensor([3, 1]), tensor([3, 1]), tensor([3, 1]), tensor([3, 1]), tensor([3, 1]), tensor([1, 3]), tensor([2, 3]), tensor([3, 1]), tensor([1, 3]), tensor([1, 3]), tensor([1, 3]), tensor([1, 3]), tensor([0, 1]), tensor([3, 1]), tensor([3, 1]), tensor([1, 3]), tensor([3, 1]), tensor([3, 1]), tensor([3, 1]), tensor([3, 0]), tensor([3, 1]), tensor([3, 2]), tensor([3, 1]), tensor([3, 0]), tensor([0, 3]), tensor([1, 3]), tensor([3, 1]), tensor([1, 3]), tensor([2, 1]), tensor([3, 1]), tensor([3, 1]), tensor([3, 1]), tensor([3, 1]), tensor([3, 1]), tensor([1, 3]), tensor([3, 1]), tensor([3, 1]), tensor([3, 0]), tensor([3, 1]), tensor([3, 0]), tensor([1, 3]), tensor([3, 0]), tensor([1, 3]), tensor([3, 0]), tensor([3, 2]), tensor(

In [6]:
Ks

[tensor([[0.9487+0.j, 0.0000+0.j],
         [0.0000+0.j, 0.9487+0.j]], device='cuda:1', dtype=torch.complex128),
 tensor([[0.0000+0.j, 0.0000+0.j],
         [0.0000+0.j, 0.3162+0.j]], device='cuda:1', dtype=torch.complex128),
 tensor([[0.3162+0.j, 0.0000+0.j],
         [0.0000+0.j, 0.0000+0.j]], device='cuda:1', dtype=torch.complex128)]